<a href="https://colab.research.google.com/github/munawirmt/movie-review-classification-model-in-nlp/blob/main/movie_review_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **movie review classification model with 50k reviews**

In [40]:
import re
import numpy as np
import nltk
import joblib


In [25]:
# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [26]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [27]:
import kagglehub
import pandas as pd
import os

# 1. Download the latest version of the dataset
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

# 2. List files in the directory to find the CSV file name
print("Files in directory:", os.listdir(path))

# 3. Load the dataset into a Pandas DataFrame
# (The IMDB dataset typically contains a 'IMDB Dataset.csv' file)
csv_path = os.path.join(path, "IMDB Dataset.csv")
df = pd.read_csv(csv_path)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print("Dataset loaded successfully. Shape:", df.shape)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Files in directory: ['IMDB Dataset.csv']
Dataset loaded successfully. Shape: (50000, 2)


In [28]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


**NLP text preprocessing**

In [29]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [30]:
def preprocess_text(text, use_stemming=False):
    # Lowercase & remove HTML tags + special characters
    text = re.sub(r'<[^>]+>', ' ', text.lower())
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenization
    tokens = word_tokenize(text)

    # Remove Stopwords and apply Lemmatization or Stemming
    cleaned_tokens = []
    for word in tokens:
        if word not in stop_words:
            if use_stemming:
                cleaned_tokens.append(stemmer.stem(word))
            else:
                cleaned_tokens.append(lemmatizer.lemmatize(word))

    return " ".join(cleaned_tokens)

In [31]:
# Preprocess dataset
print("Preprocessing reviews...")
df['clean_review'] = df['review'].apply(lambda x: preprocess_text(x, use_stemming=False))

Preprocessing reviews...


In [32]:
df['clean_review']

,clean_review
0,one reviewer mentioned watching oz episode hoo...
1,wonderful little production filming technique ...
2,thought wonderful way spend time hot summer we...
3,basically family little boy jake think zombie ...
4,petter mattei love time money visually stunnin...
...,...
49995,thought movie right good job creative original...
49996,bad plot bad dialogue bad acting idiotic direc...
49997,catholic taught parochial elementary school nu...
49998,going disagree previous comment side maltin on...


**train test split**

In [33]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

**feature extraction techniques:**


*   bow , ngrams
*   word2vec (cow , skipgram) & avg word2vec



In [34]:
from sklearn.feature_extraction.text import CountVectorizer

In [35]:
# Unigram BoW
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train_raw)
X_test_bow = bow_vectorizer.transform(X_test_raw)

In [37]:
# N-Gram (Unigrams + Bigrams)
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_ngram = ngram_vectorizer.fit_transform(X_train_raw)
X_test_ngram = ngram_vectorizer.transform(X_test_raw)

**Word2Vec (CBOW & Skip-Gram) & Avg Word2Vec**

In [43]:
!pip install kagglehub nltk gensim scikit-learn tensorflow joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 58.5 MB/s eta 0:00:00


In [44]:
from gensim.models import Word2Vec

In [45]:
# Tokenize corpus for Word2Vec training
corpus_tokens = [text.split() for text in X_train_raw]

In [46]:
# 1. Continuous Bag of Words (CBOW) -> sg=0
w2v_cbow = Word2Vec(sentences=corpus_tokens, vector_size=100, window=5, min_count=2, sg=0, workers=4)


In [47]:
# 2. Skip-Gram -> sg=1
w2v_skipgram = Word2Vec(sentences=corpus_tokens, vector_size=100, window=5, min_count=2, sg=1, workers=4)


In [48]:
# Function to compute Average Word2Vec features
def get_avg_w2v(sentences, model, vector_size=100):
    features = []
    for sentence in sentences:
        words = sentence.split()
        valid_vectors = [model.wv[word] for word in words if word in model.wv]
        if valid_vectors:
            features.append(np.mean(valid_vectors, axis=0))
        else:
            features.append(np.zeros(vector_size))
    return np.array(features)

In [49]:
# Extract Average Word2Vec features using Skip-Gram model
X_train_avg_w2v = get_avg_w2v(X_train_raw, w2v_skipgram)
X_test_avg_w2v = get_avg_w2v(X_test_raw, w2v_skipgram)

**Keras Deep Learning Word Embedding (TF-IDF Vectorizer alternative for High Accuracy)**

In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

**model training and evalution**

In [51]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [52]:
models = {
    "Naive Bayes (BoW)": (MultinomialNB(), X_train_bow, X_test_bow),
    "Naive Bayes (N-Gram)": (MultinomialNB(), X_train_ngram, X_test_ngram),
    "Logistic Regression (TF-IDF)": (LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf),
    "Random Forest (Avg Word2Vec)": (RandomForestClassifier(n_estimators=100, random_state=42), X_train_avg_w2v, X_test_avg_w2v)
}

In [53]:
best_model = None
best_accuracy = 0
best_vectorizer = None

In [54]:
for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    acc = accuracy_score(y_test, preds)

    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds))
    print("-" * 50)

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        if "TF-IDF" in name:
            best_vectorizer = tfidf_vectorizer
        elif "N-Gram" in name:
            best_vectorizer = ngram_vectorizer
        elif "BoW" in name:
            best_vectorizer = bow_vectorizer

=== Naive Bayes (BoW) ===
Accuracy: 0.8495
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      5000
           1       0.85      0.85      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

--------------------------------------------------
=== Naive Bayes (N-Gram) ===
Accuracy: 0.8555
              precision    recall  f1-score   support

           0       0.86      0.85      0.85      5000
           1       0.85      0.86      0.86      5000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

--------------------------------------------------
=== Logistic Regression (TF-IDF) ===
Accuracy: 0.8947
              precision    recall  f1-score   support

           0       0.90      0.88      0.89   

**save model and vectorizer for production deployment**

In [55]:
# Save best model and feature extractor to disk
os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(best_model, "model_artifacts/best_sentiment_model.pkl")
joblib.dump(best_vectorizer, "model_artifacts/vectorizer.pkl")

['model_artifacts/vectorizer.pkl']

In [56]:
print(f"Best model saved successfully with accuracy: {best_accuracy * 100:.2f}%")

Best model saved successfully with accuracy: 89.47%


**load artifacts and run inference (for production purpose)**

In [57]:
# Production pipeline function
def predict_sentiment(review_text):
    # Load saved artifacts
    model = joblib.load("model_artifacts/best_sentiment_model.pkl")
    vectorizer = joblib.load("model_artifacts/vectorizer.pkl")

    # Preprocess incoming text
    cleaned_text = preprocess_text(review_text)

    # Transform text and predict
    vectorized_text = vectorizer.transform([cleaned_text])
    prediction = model.predict(vectorized_text)[0]
    probability = model.predict_proba(vectorized_text)[0]

    label = "Positive" if prediction == 1 else "Negative"
    confidence = probability[prediction]

    return {"sentiment": label, "confidence": f"{confidence * 100:.2f}%"}

In [58]:
# Sample test
sample_review = "This movie had incredible acting, amazing visuals, and a brilliant storyline!"
print(predict_sentiment(sample_review))

{'sentiment': 'Positive', 'confidence': '98.82%'}


**Optimize Model Performance & Threshold Tuning**

In [59]:
from sklearn.metrics import precision_recall_curve

y_probs = best_model.predict_proba(X_test_tfidf)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

In [64]:
precisions

array([0.5       , 0.50005001, 0.50010002, ..., 1.        , 1.        ,
       1.        ])

In [65]:
recalls

array([1.e+00, 1.e+00, 1.e+00, ..., 4.e-04, 2.e-04, 0.e+00])

In [66]:
thresholds

array([8.63918842e-05, 1.12047496e-04, 1.21196368e-04, ...,
       9.98944206e-01, 9.99378894e-01, 9.99407111e-01])

**Implement Input Validation & Edge Case Handling**

In [61]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993335 sha256=f43a154b8764cbba0b83ce46783db5158c5ca4a9098dfcf3343a91452a9dfe23
  Stored in directory: /root/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect


In [62]:
import langdetect

In [63]:
def production_predict(review_text):
    # 1. Edge Case: Empty or space-only input
    if not review_text or not review_text.strip():
        return {"error": "Input review cannot be empty."}

    # 2. Edge Case: Check min character/word length
    if len(review_text.split()) < 3:
        return {"error": "Review text is too short to generate a reliable sentiment."}

    # 3. Preprocess
    cleaned_text = preprocess_text(review_text)

    # Check if text became empty after removing stopwords/special chars
    if not cleaned_text.strip():
        return {"error": "Review contains no meaningful words for analysis."}

    # 4. Inference
    vec_text = best_vectorizer.transform([cleaned_text])
    prediction = best_model.predict(vec_text)[0]
    prob = best_model.predict_proba(vec_text)[0]

    confidence = float(prob[prediction])

    # 5. Low confidence flag (e.g., flag neutral/ambiguous reviews)
    if confidence < 0.60:
        sentiment_label = "Neutral / Uncertain"
    else:
        sentiment_label = "Positive" if prediction == 1 else "Negative"

    return {
        "sentiment": sentiment_label,
        "confidence": f"{confidence * 100:.2f}%",
        "raw_scores": {"negative": float(prob[0]), "positive": float(prob[1])}
    }

In [68]:
production_predict("This movie had incredible acting, amazing visuals, and a brilliant storyline!")

{'sentiment': 'Positive',
 'confidence': '98.82%',
 'raw_scores': {'negative': 0.011779043752640828,
  'positive': 0.9882209562473592}}

In [69]:
## save model artifacts
import joblib
import os

os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(best_model, "model_artifacts/best_model.pkl")
joblib.dump(best_vectorizer, "model_artifacts/vectorizer.pkl")

['model_artifacts/vectorizer.pkl']

**create requirements.txt file**

In [70]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
nltk
joblib
numpy
pandas

Writing requirements.txt


**create app.py file**

In [72]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import re

app = FastAPI()

model = joblib.load("model_artifacts/best_model.pkl")
vectorizer = joblib.load("model_artifacts/vectorizer.pkl")

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text.lower())
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

class ReviewRequest(BaseModel):
    review: str

@app.post("/predict")
def predict_sentiment(payload: ReviewRequest):
    cleaned = clean_text(payload.review)
    vec_text = vectorizer.transform([cleaned])
    pred = model.predict(vec_text)[0]
    prob = model.predict_proba(vec_text)[0][pred]
    return {
        "sentiment": "Positive" if pred == 1 else "Negative",
        "confidence": f"{prob * 100:.2f}%"
    }

Overwriting app.py


**create docker file**

In [73]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
RUN python -c "import nltk; nltk.download('stopwords'); nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('wordnet')"

COPY . .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


**install fast api**

In [74]:
!pip install fastapi uvicorn nest_asyncio

**run and test api in colab (python threading)**

In [75]:
import uvicorn
import nest_asyncio
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Background-ൽ FastAPI സെർവർ സ്റ്റാർട്ട് ചെയ്യുന്നു
thread = threading.Thread(target=run_server)
thread.start()

Exception in thread Thread-53 (run_server):


**to test whether api working or not**

In [76]:
!curl -X 'POST' \
  'http://127.0.0.1:8000/predict' \
  -H 'Content-Type: application/json' \
  -d '{"review": "This movie was absolutely amazing and wonderful!"}'

curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Couldn't connect to server


In [77]:
import uvicorn
import nest_asyncio
import threading
import time

# Jupyter/Colab-ൽ asyncio വർക്ക് ചെയ്യാൻ
nest_asyncio.apply()

def run_server():
    # app.py-ലുള്ള app എന്ന ഓബ്‌ജക്റ്റിനെ റൺ ചെയ്യുന്നു
    uvicorn.run("app:app", host="127.0.0.1", port=8000, reload=False)

# Background Thread സ്റ്റാർട്ട് ചെയ്യുന്നു
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# സെർവർ പൂർണ്ണമായി സ്റ്റാർട്ട് ആകാൻ 3 സെക്കൻഡ് വെയ്റ്റ് ചെയ്യുന്നു
time.sleep(3)
print("FastAPI Server is running on http://127.0.0.1:8000")

INFO:     Started server process [1487]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


FastAPI Server is running on http://127.0.0.1:8000


In [78]:
!curl -X 'POST' \
  'http://127.0.0.1:8000/predict' \
  -H 'Content-Type: application/json' \
  -d '{"review": "This movie was absolutely amazing and wonderful!"}'

INFO:     127.0.0.1:34024 - "POST /predict HTTP/1.1" 200 OK
{"sentiment":"Positive","confidence":"99.39%"}

In [79]:
!npx localtunnel --port 8000

⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://fruity-goats-kneel.loca.lt
INFO:     1.39.110.92:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     1.39.110.92:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     192.178.6.136:0 - "GET /robots.txt HTTP/1.1" 404 Not Found
INFO:     74.125.216.1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     192.178.6.135:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     1.39.110.92:0 - "POST /predict HTTP/1.1" 200 OK
^C


In [92]:
%%writefile app.py
import streamlit as st
import joblib
import re

# Page Configuration
st.set_page_config(page_title="Movie Review Sentiment Analyzer", page_icon="🎬")

st.title("🎬 Movie Review Sentiment Analysis")
st.write("Enter a movie review below to predict whether it is Positive or Negative!")

# Load saved artifacts
@st.cache_resource
def load_model():
    model = joblib.load("model_artifacts/best_model.pkl")
    vectorizer = joblib.load("model_artifacts/vectorizer.pkl")
    return model, vectorizer

model, vectorizer = load_model()

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text.lower())
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

# Text input box
user_review = st.text_area("Write your movie review here:", "This movie had an amazing storyline and superb acting!")

if st.button("Analyze Sentiment"):
    if user_review.strip():
        cleaned = clean_text(user_review)
        vec_text = vectorizer.transform([cleaned])

        prediction = model.predict(vec_text)[0]
        prob = model.predict_proba(vec_text)[0][prediction]

        if prediction == 1:
            st.success(f"**Sentiment:** Positive 😊 (Confidence: {prob * 100:.2f}%)")
        else:
            st.error(f"**Sentiment:** Negative 😞 (Confidence: {prob * 100:.2f}%)")
    else:
        st.warning("Please enter a review to analyze.")

Overwriting app.py
